In [13]:
import os
os.chdir("/orcd/archive/abugoot/001/Projects/paolo/main_tde/")

import hydra
import argparse
from omegaconf import DictConfig, OmegaConf
import matplotlib.pyplot as plt
import numpy as np
import torch
import logging
from utils.seed import seed_everything  # Import seeding utility
from torch.utils.data import DataLoader

# Set random seed for reproducibility
RANDOM_SEED = 42
seed_everything(RANDOM_SEED, deterministic=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [14]:
print(device)

cuda


In [15]:
def load_all(ckpt_dir):
    # Resolve and validate checkpoint directory
    experiment_dir = os.path.abspath(os.path.expanduser(str(ckpt_dir)))
    cfg_path = os.path.join(experiment_dir, 'config.yaml')
    ckpt_path = os.path.join(experiment_dir, 'best_model.pt')
    # Load trained config and use it as the active config
    cfg = OmegaConf.load(cfg_path)

    encoder = hydra.utils.instantiate(cfg.encoder).to(device)
    generator = hydra.utils.instantiate(cfg.generator).to(device)

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    encoder.load_state_dict(checkpoint['encoder_state_dict'])
    generator.load_state_dict(checkpoint['generator_state_dict'])
    encoder.eval()
    generator.eval()
    
    dataset = hydra.utils.instantiate(cfg.dataset)

    return cfg, ckpt_path, encoder, generator, dataset



In [16]:
ckpt_dir = "outputs/virus_synthetic_5103f9a02ec3864bfd0fad977059e61b"
cfg, ckpt_path, encoder, generator, dataset = load_all(ckpt_dir)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of TimeAwareEsmForFlow were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['condition_proj.0.bias', 'condition_proj.0.weight', 'condition_proj.2.bias', 'condition_proj.2.weight', 'cross_attn_layers.0.attn.in_proj_bias', 'cross_attn_layers.0.attn.in_proj_weight', 'cross_attn_layers.0.attn.out_proj.bias', 'cross_attn_layers.0.attn.out_proj.weight', 'cross_attn_layers.0.ff.0.bias', 'cross_attn_layers.0.ff.0.weight', 'cross_attn_layers.0.ff.2.bias', 'cross_attn_layers.0.ff.2.weight', 'cross_attn_layers.0.ln1.bias', 'cross_attn_layers.0.ln1.weight', 'cross_attn_layers.0.ln2.bias', 'cross_attn_layers.0.ln2.weight', 'cross_attn_scales.0', 'cross_ff_s

In [17]:
idx = 0
print(dataset[idx]["source_samples"]["esm_input_ids"].shape)



torch.Size([16, 12])


In [ ]:
loader = DataLoader(dataset, batch_size=1)
batch = next(iter(loader))

# For dictionary samples (like PubMed dataset), move tensors to device
source_samples = {}
target_samples = {}

for key, value in batch['source_samples'].items():
    if isinstance(value, torch.Tensor):
        source_samples[key] = value.to(device)
    else:
        source_samples[key] = value
        
for key, value in batch['target_samples'].items():
    if isinstance(value, torch.Tensor):
        target_samples[key] = value.to(device)
    else:
        target_samples[key] = value
        
#x_source = x_samples["source_samples"]
#x_target = x_samples["target_samples"]

latent_source = encoder(source_samples)
latent_target = encoder(target_samples)


_, texts = generator.sample(source_samples, latent_source, latent_target, return_texts = True)


In [ ]:
print(texts)

[['ADHTDDHIMA']]


: 